# Testing v2 algorithm on unseen test datasets

This notebook runs the same end-to-end deduplication workflow on the **Depression** and **Diabetes**  datasets.

The workflow is intentionally narrow:

1. Load the dataset.
2. Build blocked candidate pairs.
3. Score the pairs with the weighted deduper.
4. Evaluate at the pair level and record level.


## Evaluation strategy

We use two complementary evaluation levels:

1. **Pair-level** — classic precision/recall/F1 on the set of blocked candidate pairs; computed for a range of score thresholds.
2. **Record-level (ASySD-style)** — following Hair et al. (2023), pairs are clustered into connected components and each cluster retains exactly one record. The resulting kept/removed decisions are compared against gold-standard duplicate groups to build a record-level confusion matrix.

The record-level view is more directly interpretable: a false positive means a real unique paper was accidentally discarded, and a false negative means a real duplicate survived deduplication.

## Setup

Imports cover three areas:

- **Standard data science stack** (`pandas`, `numpy`, `sklearn`) for data handling and computing evaluation metrics.
- **App modules** — `Deduper` is the main deduplication class; `BLOCK_RULES` defines which field combinations are used for candidate-pair blocking; `GoldStandardPaper` is the Pydantic model that adds gold-standard fields (`recordid`, `duplicateid`) on top of the base `Paper` schema.
- **Path configuration** — the notebook resolves the repository root dynamically so it can be run from either the `notebooks/` folder or the repo root.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path
import random

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from tqdm import tqdm

# Ensure imports like "from app..." work when notebook is opened from notebooks/
repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.dedupe import Deduper
from app.determine_weights import BLOCK_RULES

DATA_PATH = repo_root / "notebooks/data/diabetes_data.csv"
RESULTS_ROOT = repo_root / "notebooks" / "results"
GLOBAL_SEED = 42

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)


In [2]:
import random
import sys
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, confusion_matrix, roc_auc_score
from tqdm import tqdm

repo_root = Path.cwd()
if not (repo_root / "app").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from app.dedupe import Deduper, INTERCEPT, WEIGHTS as MODEL_WEIGHTS
from app.determine_weights import (
    BLOCK_RULES,
    build_blocked_pairs,
    build_record_cache,
    read_process_data_from_file,
)

DATA_PATH = repo_root / "notebooks" / "data" / "srsr_data.csv"
RESULTS_ROOT = repo_root / "notebooks" / "results"
GLOBAL_SEED = 42
MAX_PAIRS = None
THRESHOLDS = [0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

WEIGHTS_FILTERED = {k: v for k, v in MODEL_WEIGHTS.items() if k not in ("volume", "abstract")}
SCORE_FIELDS = list(WEIGHTS_FILTERED.keys())


def score_pairs_weighted(df: pd.DataFrame, pairs_df: pd.DataFrame) -> pd.DataFrame:
    sample_ids = set(pairs_df["id_a"]).union(pairs_df["id_b"])
    record_cache, validation_errors = build_record_cache(
        df, id_column="recordid", include_ids=sample_ids
    )

    if validation_errors:
        print(f"Skipped {validation_errors} invalid records while building the cache")

    if not record_cache:
        return pd.DataFrame(columns=["id_a", "id_b", "is_dupe", "prob", "early_stop"])

    any_paper = next(iter(record_cache.values()))
    deduper = Deduper(reference=any_paper, candidates=[any_paper])
    
    results = []
    for row in tqdm(pairs_df.itertuples(index=False), total=len(pairs_df)):
        rec_a = record_cache.get(int(row.id_a))
        rec_b = record_cache.get(int(row.id_b))
        if rec_a is None or rec_b is None:
            continue

        probability, field_scores, early_stop = deduper.score_pair(
            rec_a,
            rec_b,
            weights=WEIGHTS_FILTERED,
            intercept=INTERCEPT,
            fields=SCORE_FIELDS,
        )
        result = {
            "id_a": row.id_a,
            "id_b": row.id_b,
            "is_dupe": row.is_dupe,
            "prob": probability,
            "early_stop": early_stop,
        }
        for field in SCORE_FIELDS:
            result[f"score_{field}"] = field_scores.get(field)
        results.append(result)

    return pd.DataFrame(results)


def metrics_for_threshold(scored_df: pd.DataFrame, threshold: float) -> dict:
    y_true = scored_df["is_dupe"].astype(int)
    y_prob = scored_df["prob"].astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0

    return {
        "threshold": threshold,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "average_precision": float(average_precision_score(y_true, y_prob)),
    }


def _pair_side_by_side_export(
    df: pd.DataFrame,
    pair_df: pd.DataFrame,
    fields: list[str],
    out_path: Path,
) -> Path:
    meta_cols = [
        column
        for column in ["prob", "early_stop", *[f"score_{field}" for field in SCORE_FIELDS]]
        if column in pair_df.columns
    ]
    left = df[["recordid", *fields]].copy()
    left.columns = ["id_a", *[f"{field}_a" for field in fields]]
    right = df[["recordid", *fields]].copy()
    right.columns = ["id_b", *[f"{field}_b" for field in fields]]

    export = pair_df[["id_a", "id_b", *meta_cols]].merge(left, on="id_a", how="left").merge(right, on="id_b", how="left")
    side_by_side = ["id_a", "id_b", *meta_cols]
    for field in fields:
        side_by_side.extend([f"{field}_a", f"{field}_b"])

    export_subset = export[[column for column in side_by_side if column in export.columns]]
    out_path.parent.mkdir(parents=True, exist_ok=True)
    export_subset.to_csv(out_path, index=False)
    return out_path


def export_false_positives_side_by_side(
    df: pd.DataFrame,
    scored_df: pd.DataFrame,
    fields: list[str],
    threshold: float,
    out_path: Path,
) -> Path:
    y_true = scored_df["is_dupe"].astype(int)
    y_pred = (scored_df["prob"].astype(float) >= threshold).astype(int)
    fp_df = scored_df[(y_true == 0) & (y_pred == 1)].copy()
    return _pair_side_by_side_export(df, fp_df, fields, out_path)


def export_false_negatives_side_by_side(
    df: pd.DataFrame,
    scored_df: pd.DataFrame,
    fields: list[str],
    threshold: float,
    out_path: Path,
) -> Path:
    y_true = scored_df["is_dupe"].astype(int)
    y_pred = (scored_df["prob"].astype(float) >= threshold).astype(int)
    fn_df = scored_df[(y_true == 1) & (y_pred == 0)].copy()
    return _pair_side_by_side_export(df, fn_df, fields, out_path)


def find_best_threshold(metrics_df: pd.DataFrame, min_sensitivity: float = 0.99):
    filtered = metrics_df[metrics_df["sensitivity"] >= min_sensitivity]
    if filtered.empty:
        return None
    return filtered.loc[filtered["specificity"].idxmax()]


def _cluster_components(record_ids: Iterable[int], edges: np.ndarray) -> dict[int, int]:
    parent = {int(record_id): int(record_id) for record_id in record_ids}

    def find(node_id: int) -> int:
        while parent[node_id] != node_id:
            parent[node_id] = parent[parent[node_id]]
            node_id = parent[node_id]
        return node_id

    def union(left: int, right: int) -> None:
        root_left = find(left)
        root_right = find(right)
        if root_left != root_right:
            parent[root_right] = root_left

    for left, right in edges:
        union(int(left), int(right))

    root_to_group: dict[int, int] = {}
    record_to_group: dict[int, int] = {}
    for record_id in parent:
        root = find(record_id)
        group_id = root_to_group.setdefault(root, len(root_to_group))
        record_to_group[record_id] = group_id

    return record_to_group


def record_level_metrics_for_threshold(
    df_orig: pd.DataFrame,
    scored_pairs_df: pd.DataFrame,
    threshold: float,
) -> dict:
    df_t = df_orig.copy()
    edges = scored_pairs_df[scored_pairs_df["prob"] >= threshold][["id_a", "id_b"]].to_numpy()
    record_to_group = _cluster_components(df_t["recordid"].tolist(), edges)

    df_t["predicted_group"] = df_t["recordid"].map(record_to_group)
    df_t["pred_keep"] = df_t.groupby("predicted_group")["recordid"].transform("min") == df_t["recordid"]

    df_t["true_cluster"] = df_t["duplicateid"].fillna(df_t["recordid"].astype(str)).astype(str)
    df_t["true_keep"] = df_t.groupby("true_cluster")["recordid"].transform("min") == df_t["recordid"]

    true_removed = ~df_t["true_keep"]
    pred_removed = ~df_t["pred_keep"]

    tp = int((true_removed & pred_removed).sum())
    fp = int((~true_removed & pred_removed).sum())
    tn = int((~true_removed & ~pred_removed).sum())
    fn = int((true_removed & ~pred_removed).sum())

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0

    return {
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
    }

In [6]:
import sys
from loguru import logger
logger.remove()
logger.add(sys.stderr, level="WARNING")

DATASETS = {
    'diabetes': repo_root / 'notebooks' / 'data' / 'diabetes_data.csv',
    'depression': repo_root / 'notebooks' / 'data' / 'depression_data.csv'
}

def _print_threshold_diagnostics(metrics_df: pd.DataFrame, level_name: str, min_sensitivity: float = 0.98) -> None:
    if metrics_df.empty:
        print(f"No {level_name} metrics were computed.")
        return

    max_sens_row = metrics_df.loc[metrics_df['sensitivity'].idxmax()]
    max_spec_row = metrics_df.loc[metrics_df['specificity'].idxmax()]
    closest_row = metrics_df.iloc[(metrics_df['sensitivity'] - min_sensitivity).abs().argsort().iloc[0]]

    print(
        f"No {level_name} threshold reached sensitivity >= {min_sensitivity:.2f}. "
        f"Best observed sensitivity was {max_sens_row['sensitivity']:.4f} at threshold {max_sens_row['threshold']:.2f}."
    )
    print(
        f"Closest to target: threshold {closest_row['threshold']:.2f} "
        f"(sensitivity={closest_row['sensitivity']:.4f}, specificity={closest_row['specificity']:.4f})."
    )
    print(
        f"Highest specificity occurred at threshold {max_spec_row['threshold']:.2f} "
        f"(specificity={max_spec_row['specificity']:.4f}, sensitivity={max_spec_row['sensitivity']:.4f})."
    )

def run_dataset_pipeline(dataset_name: str, data_path: Path, max_pairs: int | None = None) -> dict:
    print(f'\n=== {dataset_name.upper()} ===')
    df = read_process_data_from_file(data_path, encoding='latin-1')
    all_pairs_df = build_blocked_pairs(
        df,
        block_rules=BLOCK_RULES,
        id_column='recordid',
        dup_column='duplicateid',
    )

    if max_pairs is not None and len(all_pairs_df) > max_pairs:
        pairs_df = all_pairs_df.sample(n=max_pairs, random_state=GLOBAL_SEED).reset_index(drop=True)
    else:
        pairs_df = all_pairs_df.copy().reset_index(drop=True)

    scored_df = score_pairs_weighted(df, pairs_df)

    dataset_dir = RESULTS_ROOT / dataset_name
    dataset_dir.mkdir(parents=True, exist_ok=True)

    pairs_df.to_csv(dataset_dir / f'pairs_seed_{GLOBAL_SEED}.csv', index=False)
    scored_df.to_csv(dataset_dir / f'scored_pairs_seed_{GLOBAL_SEED}.csv', index=False)

    pair_metrics_rows = []
    for threshold in THRESHOLDS:
        pair_metrics_rows.append(metrics_for_threshold(scored_df, threshold))
        export_false_positives_side_by_side(df, scored_df, SCORE_FIELDS, threshold, dataset_dir / f'false_positives_{threshold}.csv')
        export_false_negatives_side_by_side(df, scored_df, SCORE_FIELDS, threshold, dataset_dir / f'false_negatives_{threshold}.csv')

    pair_metrics_df = pd.DataFrame(pair_metrics_rows)
    pair_metrics_df.to_csv(dataset_dir / 'pair_metrics.csv', index=False)

    record_metrics_rows = [record_level_metrics_for_threshold(df, scored_df, threshold) for threshold in THRESHOLDS]
    record_metrics_df = pd.DataFrame(record_metrics_rows)
    record_metrics_df.to_csv(dataset_dir / 'record_metrics.csv', index=False)

    # Objective: maximize specificity (minimize false positives) with sensitivity floor.
    min_sensitivity = 0.99
    best_pair = find_best_threshold(pair_metrics_df, min_sensitivity=min_sensitivity)
    best_record = find_best_threshold(record_metrics_df, min_sensitivity=min_sensitivity)

    if best_pair is not None:
        print(f"Best pair-level threshold (max specificity, sens >= {min_sensitivity:.2f}): {best_pair['threshold']:.2f} | sensitivity={best_pair['sensitivity']:.4f} | specificity={best_pair['specificity']:.4f}")
    else:
        _print_threshold_diagnostics(pair_metrics_df, level_name='pair-level', min_sensitivity=min_sensitivity)

    if best_record is not None:
        print(f"Best record-level threshold (max specificity, sens >= {min_sensitivity:.2f}): {best_record['threshold']:.2f} | sensitivity={best_record['sensitivity']:.4f} | specificity={best_record['specificity']:.4f}")
    else:
        _print_threshold_diagnostics(record_metrics_df, level_name='record-level', min_sensitivity=min_sensitivity)

    return {
        'dataset': dataset_name,
        'df': df,
        'pairs_df': pairs_df,
        'scored_df': scored_df,
        'pair_metrics_df': pair_metrics_df,
        'record_metrics_df': record_metrics_df,
        'best_pair': best_pair,
        'best_record': best_record,
    }

dataset_runs = {name: run_dataset_pipeline(name, path) for name, path in DATASETS.items()}

summary_df = pd.DataFrame([
    {
        'dataset': dataset_name,
        'n_records': len(result['df']),
        'n_pairs': len(result['pairs_df']),
        'pair_best_threshold': None if result['best_pair'] is None else result['best_pair']['threshold'],
        'pair_best_sensitivity': None if result['best_pair'] is None else result['best_pair']['sensitivity'],
        'pair_best_specificity': None if result['best_pair'] is None else result['best_pair']['specificity'],
        'record_best_threshold': None if result['best_record'] is None else result['best_record']['threshold'],
        'record_best_sensitivity': None if result['best_record'] is None else result['best_record']['sensitivity'],
        'record_best_specificity': None if result['best_record'] is None else result['best_record']['specificity'],
    }
    for dataset_name, result in dataset_runs.items()
])

summary_df


=== DIABETES ===


100%|██████████| 8487/8487 [00:00<00:00, 14616.91it/s]


Best pair-level threshold (max specificity, sens >= 0.99): 0.70 | sensitivity=0.9902 | specificity=0.9949
Best record-level threshold (max specificity, sens >= 0.99): 0.85 | sensitivity=0.9921 | specificity=0.9983

=== DEPRESSION ===


c:\Users\qtnzkh4\OneDrive - University College London\deduplication-toolkit-now\app\determine_weights.py:176: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  refdata = pd.read_csv(filepath, **kwargs)
  1%|          | 6543/871718 [00:00<01:50, 7825.98it/s]2026-07-13 14:49:57.570 | WARNING  | app.dedupe:compare_authors:614 - One or both records have no authors.
2026-07-13 14:49:57.570 | WARNING  | app.dedupe:compare_authors:614 - One or both records have no authors.
2026-07-13 14:49:57.570 | WARNING  | app.dedupe:compare_authors:614 - One or both records have no authors.
2026-07-13 14:49:57.585 | WARNING  | app.dedupe:compare_authors:614 - One or both records have no authors.
  1%|          | 7328/871718 [00:00<01:55, 7498.44it/s]2026-07-13 14:49:57.649 | WARNING  | app.dedupe:compare_authors:614 - One or both records have no authors.
2026-07-13 14:49:57.664 | WARNING  | app.dedupe:compare_authors:614 - One or both records have no au

Best pair-level threshold (max specificity, sens >= 0.99): 0.80 | sensitivity=0.9939 | specificity=0.9999
Best record-level threshold (max specificity, sens >= 0.99): 0.80 | sensitivity=0.9901 | specificity=0.9993


,dataset,n_records,n_pairs,pair_best_threshold,pair_best_sensitivity,pair_best_specificity,record_best_threshold,record_best_sensitivity,record_best_specificity
0,diabetes,1845,8487,0.7,0.990215,0.994943,0.85,0.992070,0.998288
1,depression,79880,871718,0.8,0.993909,0.999925,0.80,0.990133,0.999326


In [7]:
MIN_SENSITIVITY = 0.98
THRESHOLD_OVERRIDE = 0.85  # Set to a float like 0.75 to force one threshold for all datasets.

if 'dataset_runs' not in globals():
    raise RuntimeError('Run Cell 5 first to generate dataset_runs.')

final_record_level_rows = []
for dataset_name, result in dataset_runs.items():
    record_metrics_df = result['record_metrics_df'].copy()

    if THRESHOLD_OVERRIDE is not None:
        matched = record_metrics_df[record_metrics_df['threshold'] == THRESHOLD_OVERRIDE]
        if matched.empty:
            raise ValueError(
                f'Threshold {THRESHOLD_OVERRIDE} not found in record metrics for {dataset_name}. '
                f'Available thresholds: {sorted(record_metrics_df["threshold"].tolist())}'
            )
        chosen = matched.iloc[0]
        selection_note = 'manual_threshold_override'
    else:
        chosen = find_best_threshold(record_metrics_df, min_sensitivity=MIN_SENSITIVITY)
        if chosen is None:
            chosen = record_metrics_df.loc[record_metrics_df['sensitivity'].idxmax()]
            selection_note = f"fallback_max_sensitivity_below_{MIN_SENSITIVITY:.2f}"
        else:
            selection_note = 'max_specificity_with_sensitivity_floor'

    final_record_level_rows.append({
        'dataset': dataset_name,
        'threshold': float(chosen['threshold']),
        'TP': int(chosen['TP']),
        'FP': int(chosen['FP']),
        'TN': int(chosen['TN']),
        'FN': int(chosen['FN']),
        'sensitivity': float(chosen['sensitivity']),
        'specificity': float(chosen['specificity']),
        'precision': float(chosen['precision']),
        'selection_rule': selection_note,
    })

final_record_level_df = pd.DataFrame(final_record_level_rows).sort_values(['threshold', 'dataset']).reset_index(drop=True)
final_record_level_df = final_record_level_df[[
    'dataset',
    'threshold',
    'TP',
    'FP',
    'TN',
    'FN',
    'sensitivity',
    'specificity',
    'precision',
    'selection_rule',
]]
final_record_level_df

,dataset,threshold,TP,FP,TN,FN,sensitivity,specificity,precision,selection_rule
0,depression,0.85,9979,39,69706,156,0.984608,0.999441,0.996107,manual_threshold_override
1,diabetes,0.85,1251,1,583,10,0.992070,0.998288,0.999201,manual_threshold_override


In [8]:
# Manual FP inspection table: all pair-level false positives at selected threshold(s)
INSPECT_FIELDS = [
    'title', 'authors', 'year', 'journal', 'volume', 'issue', 'pages', 'doi', 'abstract'
]
FP_EXPORT_PATH = RESULTS_ROOT / 'test_test_false_positives_manual_review.csv'

if 'dataset_runs' not in globals():
    raise RuntimeError('Run Cell 5 first to generate dataset_runs.')

threshold_by_dataset = {}
for dataset_name, result in dataset_runs.items():
    record_metrics_df = result['record_metrics_df'].copy()

    if 'THRESHOLD_OVERRIDE' in globals() and THRESHOLD_OVERRIDE is not None:
        threshold = float(THRESHOLD_OVERRIDE)
    else:
        min_sens = MIN_SENSITIVITY if 'MIN_SENSITIVITY' in globals() else 0.98
        chosen = find_best_threshold(record_metrics_df, min_sensitivity=min_sens)
        if chosen is None:
            chosen = record_metrics_df.loc[record_metrics_df['sensitivity'].idxmax()]
        threshold = float(chosen['threshold'])

    threshold_by_dataset[dataset_name] = threshold

fp_rows = []
for dataset_name, result in dataset_runs.items():
    threshold = threshold_by_dataset[dataset_name]
    scored_df = result['scored_df'].copy()
    df = result['df'].copy()

    pred = (scored_df['prob'].astype(float) >= threshold).astype(int)
    fp_df = scored_df[(scored_df['is_dupe'].astype(int) == 0) & (pred == 1)].copy()

    if fp_df.empty:
        continue

    left_cols = ['recordid', *[c for c in INSPECT_FIELDS if c in df.columns]]
    right_cols = ['recordid', *[c for c in INSPECT_FIELDS if c in df.columns]]

    left = df[left_cols].copy()
    left.columns = ['id_a', *[f'{c}_a' for c in left_cols[1:]]]
    right = df[right_cols].copy()
    right.columns = ['id_b', *[f'{c}_b' for c in right_cols[1:]]]

    merged = fp_df.merge(left, on='id_a', how='left').merge(right, on='id_b', how='left')
    merged.insert(0, 'dataset', dataset_name)
    merged.insert(1, 'threshold_used', threshold)
    fp_rows.append(merged)

if fp_rows:
    all_false_positives_df = pd.concat(fp_rows, ignore_index=True)
    side_by_side_cols = []
    for field in [c for c in INSPECT_FIELDS if f'{c}_a' in all_false_positives_df.columns and f'{c}_b' in all_false_positives_df.columns]:
        side_by_side_cols.extend([f'{field}_a', f'{field}_b'])

    lead_cols = [
        c for c in ['dataset', 'threshold_used', 'id_a', 'id_b', 'prob', 'early_stop', 'is_dupe']
        if c in all_false_positives_df.columns
    ]
    other_cols = [c for c in all_false_positives_df.columns if c not in lead_cols + side_by_side_cols]
    all_false_positives_df = all_false_positives_df[lead_cols + side_by_side_cols + other_cols]
    all_false_positives_df = all_false_positives_df.sort_values(['dataset', 'prob', 'id_a', 'id_b'], ascending=[True, False, True, True]).reset_index(drop=True)
else:
    all_false_positives_df = pd.DataFrame(columns=['dataset', 'threshold_used', 'id_a', 'id_b', 'prob'])

FP_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
all_false_positives_df.to_csv(FP_EXPORT_PATH, index=False)

print(f'Total false-positive pairs for manual inspection: {len(all_false_positives_df)}')
print(f'Exported CSV: {FP_EXPORT_PATH}')
all_false_positives_df

Total false-positive pairs for manual inspection: 59
Exported CSV: c:\Users\qtnzkh4\OneDrive - University College London\deduplication-toolkit-now\notebooks\results\test_test_false_positives_manual_review.csv


,dataset,threshold_used,id_a,id_b,prob,early_stop,is_dupe,title_a,title_b,authors_a,...,doi_b,abstract_a,abstract_b,score_doi,score_title,score_authors,score_year,score_journal,score_pages,score_issue
0,depression,0.85,43882,43883,0.999944,None,0,Kinin B1 receptors mediate depression-like beh...,Kinin B<inf>1 </inf>receptors mediate depressi...,VianaA.F.MacielI.S.DornellesF.N.FigueiredoC.P....,...,http://dx.doi.org/10.1186/1742-2094-7-98,BACKGROUND: Kinin B1 receptors are inducible m...,Background: Kinin B<inf>1 </inf>receptors are ...,1.0,0.919708,1.000000,1.0,0.746190,1.0,NaN
1,depression,0.85,43882,76312,0.999944,None,0,Kinin B1 receptors mediate depression-like beh...,Kinin B<inf>1 </inf>receptors mediate depressi...,VianaA.F.MacielI.S.DornellesF.N.FigueiredoC.P....,...,http://dx.doi.org/10.1186/1742-2094-7-98,BACKGROUND: Kinin B1 receptors are inducible m...,Background: Kinin B<inf>1 </inf>receptors are ...,1.0,0.919708,1.000000,1.0,0.746190,1.0,NaN
2,depression,0.85,43882,76313,0.999944,None,0,Kinin B1 receptors mediate depression-like beh...,Kinin B<inf>1 </inf>receptors mediate depressi...,VianaA.F.MacielI.S.DornellesF.N.FigueiredoC.P....,...,http://dx.doi.org/10.1186/1742-2094-7-98,BACKGROUND: Kinin B1 receptors are inducible m...,Background: Kinin B<inf>1 </inf>receptors are ...,1.0,0.919708,1.000000,1.0,0.746190,1.0,NaN
3,depression,0.85,43882,76314,0.999944,None,0,Kinin B1 receptors mediate depression-like beh...,Kinin B<inf>1 </inf>receptors mediate depressi...,VianaA.F.MacielI.S.DornellesF.N.FigueiredoC.P....,...,http://dx.doi.org/10.1186/1742-2094-7-98,BACKGROUND: Kinin B1 receptors are inducible m...,Background: Kinin B<inf>1 </inf>receptors are ...,1.0,0.919708,1.000000,1.0,0.746190,1.0,NaN
4,depression,0.85,33440,33441,0.999938,None,0,Fractalkine receptor (CX3CR1) deficiency sensi...,Fractalkine receptor (CX<inf>3</inf>CR1) defic...,CoronaA.W.HuangY.O'ConnorJ.C.DantzerR.KelleyK....,...,http://dx.doi.org/10.1186/1742-2094-7-93,BACKGROUND: Interactions between fractalkine (...,Background: Interactions between fractalkine (...,1.0,0.910569,1.000000,1.0,0.746190,1.0,NaN
5,depression,0.85,33440,75174,0.999938,None,0,Fractalkine receptor (CX3CR1) deficiency sensi...,Fractalkine receptor (CX<inf>3</inf>CR1) defic...,CoronaA.W.HuangY.O'ConnorJ.C.DantzerR.KelleyK....,...,http://dx.doi.org/10.1186/1742-2094-7-93,BACKGROUND: Interactions between fractalkine (...,Background: Interactions between fractalkine (...,1.0,0.910569,1.000000,1.0,0.746190,1.0,NaN
6,depression,0.85,33440,75175,0.999938,None,0,Fractalkine receptor (CX3CR1) deficiency sensi...,Fractalkine receptor (CX<inf>3</inf>CR1) defic...,CoronaA.W.HuangY.O'ConnorJ.C.DantzerR.KelleyK....,...,http://dx.doi.org/10.1186/1742-2094-7-93,BACKGROUND: Interactions between fractalkine (...,Background: Interactions between fractalkine (...,1.0,0.910569,1.000000,1.0,0.746190,1.0,NaN
7,depression,0.85,33440,75176,0.999938,None,0,Fractalkine receptor (CX3CR1) deficiency sensi...,Fractalkine receptor (CX<inf>3</inf>CR1) defic...,CoronaA.W.HuangY.O'ConnorJ.C.DantzerR.KelleyK....,...,http://dx.doi.org/10.1186/1742-2094-7-93,BACKGROUND: Interactions between fractalkine (...,Background: Interactions between fractalkine (...,1.0,0.910569,1.000000,1.0,0.746190,1.0,NaN
8,depression,0.85,61295,78756,0.998814,None,0,[Role of peripheral adrenergic links in the de...,Role of peripheral adrenergic links in the dev...,KhaismanE.B.KushnarevV.V.,...,NaN,The study was made on the reserpine model of b...,The study was made on the reserpine model of b...,NaN,0.985612,1.000000,1.0,0.850000,1.0,1.000000
9,depression,0.85,69275,69281,0.997717,None,0,Uptake of5 hydroxytryptamine andnoradrenaline ...,Uptake of (3H)5-hydroxytryptamine and (3H)nora...,TuckJ.R.PunellG.,...,NaN,NaN,NaN,NaN,0.928994,1.000000,1.0,0.850000,1.0,1.000000


In [7]:
print("INTERCEPT:", INTERCEPT)
print("WEIGHTS_FILTERED keys:", list(WEIGHTS_FILTERED.keys()))
print("WEIGHTS_FILTERED summary:", {k: WEIGHTS_FILTERED[k] for k in list(WEIGHTS_FILTERED)[:5]})

for name, res in dataset_runs.items():
    s = res['scored_df']
    early_stop_rate = (s['early_stop'].notna().mean() if 'early_stop' in s.columns else float('nan'))
    print(f"{name}: early_stop_rate={early_stop_rate:.4f}")

INTERCEPT: -18.68666
WEIGHTS_FILTERED keys: ['doi', 'title', 'authors', 'year', 'journal', 'pages', 'issue']
WEIGHTS_FILTERED summary: {'doi': 5.30393, 'title': 11.591455, 'authors': 3.702789, 'year': 2.645679, 'journal': 3.179708}
srsr: early_stop_rate=0.9773
cardiac: early_stop_rate=0.9384
neuroimaging: early_stop_rate=0.7996
